<a href="https://colab.research.google.com/github/Inan404/SLM-Paper/blob/main/SLM_Paper_Reproduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Downloading SLMS, Setup Ollama and Langchain


In [2]:

!curl -fsSL https://ollama.com/install.sh | sh


import subprocess
import time

ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


print("Starting Ollama server...")
time.sleep(5)
print("✓ Ollama server started!")

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama server...
✓ Ollama server started!


In [3]:
!pip install -q langchain langchain-ollama

!ollama pull phi4:14b
# ollama pull llama3.2:3b
# ollama pull gemma2:9b
# ollama pull deepseek-r1:14b

!ollama list



NAME        ID              SIZE      MODIFIED               
phi4:14b    ac896e5b8b34    9.1 GB    Less than a second ago    


# Basic Testing

In [4]:
from langchain_ollama import OllamaLLM

model = OllamaLLM(model="phi4:14b")


test_prompt = """You are a competitive programmer.
Generate only Python code (no explanations) to solve:

Problem: Read an integer n and print all numbers from 1 to n, one per line.
Input: Single integer n (1 ≤ n ≤ 100)
Output: Numbers from 1 to n, each on new line
"""

print("Generating solution...")
response = model.invoke(test_prompt)
print("\nGenerated code:")
print("=" * 50)
print(response)

Generating solution...

Generated code:
```python
n = int(input())
for i in range(1, n + 1):
    print(i)
```


In [5]:
!    ollama ps

NAME        ID              SIZE     PROCESSOR    CONTEXT    UNTIL              
phi4:14b    ac896e5b8b34    10 GB    100% GPU     4096       4 minutes from now    


In [6]:
import re
import subprocess
import os

class CodeforcesSolver:
    def __init__(self, model_name="phi4:14b"):
        self.model = OllamaLLM(model=model_name)
        self.model_name = model_name

    def create_prompt(self, problem_statement, language="Python"):
        """Create prompt following the paper's methodology"""
        template = f"""You are a highly skilled competitive programmer with 15 years of experience in the field.

Your objective is to analyze the following problem statement and to produce a {language} code solution that adheres to the requirements.

Guidelines for the solution:
- Deliver only the {language} code
- Ensure the solution reads input via standard input and produces outputs via standard output
- If the solution requires defining a function, ensure it is executed within the code
- Avoid adding explanations, comments, or unnecessary text

The Problem Statement includes a detailed description, input and output format, and examples to clarify requirements.

Problem Statement:
{problem_statement}

Generate the {language} solution:"""
        return template

    def extract_code(self, model_output):
        """Extract code from model output"""
        pattern = r'```(?:python|cpp|c\+\+)?\n(.*?)```'
        matches = re.findall(pattern, model_output, re.DOTALL)
        if matches:
            return matches[0].strip()
        return model_output.strip()

    def generate_solutions(self, problem_statement, language="Python", num_attempts=3):
        """Generate multiple solutions (pass@k)"""
        prompt = self.create_prompt(problem_statement, language)
        solutions = []

        for i in range(num_attempts):
            print(f"Generating solution {i+1}/{num_attempts}...")
            response = self.model.invoke(prompt)
            clean_code = self.extract_code(response)
            solutions.append(clean_code)

        return solutions

    def test_solution(self, code, test_cases):
        """Test solution with sample test cases"""
        results = []

        for i, (test_input, expected_output) in enumerate(test_cases):
            with open('temp_solution.py', 'w') as f:
                f.write(code)

            try:
                process = subprocess.run(
                    ['python', 'temp_solution.py'],
                    input=test_input,
                    capture_output=True,
                    text=True,
                    timeout=5
                )

                actual_output = process.stdout.strip()
                expected = expected_output.strip()
                passed = actual_output == expected

                results.append({
                    'test_case': i+1,
                    'passed': passed,
                    'expected': expected,
                    'actual': actual_output
                })

            except subprocess.TimeoutExpired:
                results.append({
                    'test_case': i+1,
                    'passed': False,
                    'error': 'Time Limit Exceeded'
                })
            except Exception as e:
                results.append({
                    'test_case': i+1,
                    'passed': False,
                    'error': str(e)
                })

        return results

solver = CodeforcesSolver(model_name="phi4:14b")


In [7]:
problem = """
Problem: Watermelon

Pete and Billy want to divide a watermelon weighing w kilos into two parts,
where each part weighs an even number of kilos (but parts don't need to be equal).
Each person must get positive weight.

Input: Integer w (1 ≤ w ≤ 100) - weight of watermelon
Output: "YES" if possible to divide into two even parts, "NO" otherwise

Examples:
Input: 8
Output: YES

Note: Can divide into 2+6 or 4+4 kilos
"""

test_cases = [
    ("8\n", "YES"),
    ("2\n", "NO"),
    ("4\n", "YES"),
    ("3\n", "NO"),
]


print("=" * 60)
print("Generating solutions...")
print("=" * 60)
solutions = solver.generate_solutions(problem, "Python", num_attempts=3)

all_results = []
for i, solution in enumerate(solutions, 1):
    print(f"\n{'='*60}")
    print(f"Solution {i}")
    print('='*60)
    print(solution)
    print()

    results = solver.test_solution(solution, test_cases)
    all_results.append(results)


    passed_all = all(r['passed'] for r in results)
    for result in results:
        status = "✓ PASSED" if result['passed'] else "✗ FAILED"
        print(f"  Test {result['test_case']}: {status}")
        if not result['passed'] and 'error' not in result:
            print(f"    Expected: {result['expected']}")
            print(f"    Got: {result['actual']}")

any_passed = any(all(r['passed'] for r in results) for results in all_results)
print(f"\n{'='*60}")
print(f"Pass@3: {'✓ PASSED' if any_passed else '✗ FAILED'}")
print('='*60)

Generating solutions...
Generating solution 1/3...
Generating solution 2/3...
Generating solution 3/3...

Solution 1
w = int(input())
print("YES" if w > 2 and w % 2 == 0 else "NO")

  Test 1: ✓ PASSED
  Test 2: ✓ PASSED
  Test 3: ✓ PASSED
  Test 4: ✓ PASSED

Solution 2
def can_divide_watermelon(w):
    return 'YES' if w % 2 == 0 and w > 2 else 'NO'

if __name__ == "__main__":
    w = int(input())
    print(can_divide_watermelon(w))

  Test 1: ✓ PASSED
  Test 2: ✓ PASSED
  Test 3: ✓ PASSED
  Test 4: ✓ PASSED

Solution 3
w = int(input())
print("YES" if w > 2 and w % 2 == 0 else "NO")

  Test 1: ✓ PASSED
  Test 2: ✓ PASSED
  Test 3: ✓ PASSED
  Test 4: ✓ PASSED

Pass@3: ✓ PASSED


In [8]:

!nvidia-smi

Sat Oct 18 12:11:51 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P0             30W /   70W |    9800MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Creating Dataset

In [9]:
import torch
if torch.cuda.is_available():
    print(f"✓ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠ No GPU - using CPU (slower but works)")

✓ GPU available: Tesla T4
  Memory: 15.83 GB


In [14]:
import pandas as pd
import os

def merge_contest_index_from_xlsx(directory_path):
    """
    Reads all XLSX files in a directory, merges 'contest_id' and 'index'
    values for each row, and returns a list of merged strings.

    Args:
        directory_path (str): The path to the directory containing XLSX files.

    Returns:
        list: A list of strings where each string is the merged value
              of 'contest_id' and 'index' for a row.
    """
    merged_values = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory_path, filename)
            try:
                df = pd.read_csv(filepath)
                if 'contest_id' in df.columns and 'index' in df.columns:
                    # Iterate through rows and merge values
                    for index, row in df.iterrows():
                        contest_id = str(row['contest_id'])
                        index_val = str(row['index'])
                        merged_values.append(contest_id + index_val)
                else:
                    print(f"Warning: 'contest_id' or 'index' column not found in {filename}")
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    return merged_values


directory = '/content/drive/MyDrive/data_small_language_models/selected_problems'
merged_data = merge_contest_index_from_xlsx(directory)
print(merged_data)

['4A', '71A', '231A', '282A', '158A', '50A', '263A', '112A', '339A', '281A', '236A', '791A', '266A', '617A', '546A', '59A', '977A', '110A', '734A', '41A', '116A', '677A', '271A', '266B', '1030A', '467A', '344A', '486A', '136A', '200B', '61A', '228A', '705A', '1328A', '520A', '469A', '144A', '148A', '996A', '443A', '785A', '268A', '1335A', '510A', '141A', '1352A', '723A', '427A', '750A', '155A', '1742A', '151A', '732A', '1154A', '1703A', '96A', '160A', '318A', '133A', '405A', '208A', '580A', '337A', '451A', '1475A', '313A', '34B', '1374B', '1475B', '1343A', '460A', '474A', '556A', '977B', '149A', '320A', '1335B', '1850D', '1624B', '1373B', '1985D', '115A', '1807D', '1878C', '1337B', '1883B', '1941C', '1971C', '1927B', '1742C', '1462C', '709A', '567A', '1559A', '26A', '1339A', '1837B', '1915D', '1535B', '1537B', '275A', '1296B', '1714A', '1921C', '1992C', '1341A', '1543A', '746B', '1555A', '1872B', '25A', '4C', '230B', '189A', '451B', '459B', '478B', '1294C', '1360D', '476B', '600B', '15

In [15]:
len(merged_data)

350

In [ ]:
!pip install -q datasets pandas tqdm

from datasets import load_dataset
from tqdm.auto import tqdm
import json
import pandas as pd

my_problems = merged_data



print(f"Looking for {len(my_problems)} problems: {my_problems}\n")


print("Loading dataset in streaming mode (no download)...")
dataset = load_dataset("open-r1/codeforces", streaming=True)
print("✓ Dataset ready (streaming mode)\n")


def normalize_problem_id(pid):
    """Normalize problem ID for matching"""
    pid = str(pid).upper().strip()
    variations = {pid, pid.replace('/', '')}

    if '/' not in pid:
        for i in range(1, len(pid)):
            if pid[i].isalpha():
                variations.add(f"{pid[:i]}/{pid[i:]}")
                break

    return variations

target_lookup = {}
for pid in my_problems:
    for variant in normalize_problem_id(pid):
        target_lookup[variant] = pid

print(f"Searching for {len(target_lookup)} problem variations...\n")


found_problems = []
found_ids = set()

print("Streaming and searching...")

for problem in tqdm(dataset['train'], desc="Searching"):
    problem_id = problem.get('id', '')
    contest_id = str(problem.get('contest_id', ''))
    index = problem.get('index', '')

    variations = set()
    if problem_id:
        variations.update(normalize_problem_id(problem_id))
    if contest_id and index:
        variations.update(normalize_problem_id(f"{contest_id}{index}"))
        variations.update(normalize_problem_id(f"{contest_id}/{index}"))

    matched = variations & target_lookup.keys()

    if matched:
        original_id = target_lookup[matched.pop()]

        # Skip if already found (avoid duplicates)
        if original_id in found_ids:
            continue

        title = problem.get('title', '')
        description = problem.get('description', '')

        # Build problem statement
        if title and description:
            problem_statement = f"# {title}\n\n{description}"
        elif description:
            problem_statement = description
        elif title:
            problem_statement = title
        else:
            problem_statement = "No description available"

        # Extract test cases
        test_cases = []
        official_tests = problem.get('official_tests', [])

        if official_tests:
            for test in official_tests:
                if isinstance(test, dict):
                    test_cases.append((
                        test.get('input', ''),
                        test.get('output', '')
                    ))

        # Store result
        found_problems.append({
            'problem_id': original_id,
            'dataset_id': problem_id or f"{contest_id}/{index}",
            'problem_statement': problem_statement,
            'test_cases': test_cases,
            'num_tests': len(test_cases),
            'rating': problem.get('rating'),
            'tags': problem.get('tags', [])
        })

        found_ids.add(original_id)
        print(f"✓ Found {original_id}: {len(test_cases)} tests")

        # EARLY EXIT: Stop when all problems found
        if len(found_ids) == len(my_problems):
            print("\n✓ All problems found! Stopping stream.")
            break

not_found = [p for p in my_problems if p not in found_ids]

print(f"\n{'='*60}")
print("RESULTS")
print('='*60)
print(f"✓ Found: {len(found_problems)}/{len(my_problems)}")
print(f"✗ Not found: {len(not_found)}")

if not_found:
    print(f"\nNot found: {not_found}")

if found_problems:
    total_tests = sum(p['num_tests'] for p in found_problems)
    print(f"\nTotal test cases: {total_tests}")
    print(f"Average per problem: {total_tests/len(found_problems):.1f}")


formatted_problems = found_problems

print(f"\n✓ Variable 'formatted_problems' ready with {len(formatted_problems)} problems")


output = {
    'metadata': {
        'total_requested': len(my_problems),
        'total_found': len(found_problems),
        'total_not_found': len(not_found),
        'total_test_cases': sum(p['num_tests'] for p in formatted_problems),
        'timestamp': pd.Timestamp.now().isoformat()
    },
    'problems': formatted_problems,
    'not_found': not_found
}

with open('problems_with_testcases.json', 'w') as f:
    json.dump(output, f, indent=2)

print("✓ Saved to: problems_with_testcases.json")


if formatted_problems:
    print(f"\n{'='*60}")
    print("SAMPLE PROBLEM")
    print('='*60)

    sample = formatted_problems[0]
    print(f"\nID: {sample['problem_id']}")
    print(f"Rating: {sample['rating']}")
    print(f"Tests: {sample['num_tests']}")
    print(f"\nStatement: {sample['problem_statement'][:200]}...")

    if sample['test_cases']:
        inp, out = sample['test_cases'][0]
        print(f"\nTest 1:")
        print(f"  In:  {repr(inp[:80])}")
        print(f"  Out: {repr(out[:80])}")

if formatted_problems:
    df = pd.DataFrame([
        {'id': p['problem_id'], 'tests': p['num_tests'], 'rating': p['rating']}
        for p in formatted_problems
    ])
    print(f"\n{'='*60}")
    print("SUMMARY")
    print('='*60)
    print(df.to_string(index=False))
    df.to_csv('summary.csv', index=False)

print(f"\n{'='*60}")
print("✓ READY FOR EVALUATION!")
print('='*60)
print(f"Use: formatted_problems ({len(formatted_problems)} problems)")

In [20]:
formatted_problems[0]

{'problem_id': '295B',
 'dataset_id': '295/B',
 'problem_statement': '# Greg and Graph\n\nGreg has a weighed directed graph, consisting of n vertices. In this graph any pair of distinct vertices has an edge between them in both directions. Greg loves playing with the graph and now he has invented a new game:\n\n- The game consists of n steps.\n- On the i-th step Greg removes vertex number xi from the graph. As Greg removes a vertex, he also removes all the edges that go in and out of this vertex.\n- Before executing each step, Greg wants to know the sum of lengths of the shortest paths between all pairs of the remaining vertices. The shortest path can go through any remaining vertex. In other words, if we assume that d(i,\u2009v,\u2009u) is the shortest path between vertices v and u in the graph that formed before deleting vertex xi, then Greg wants to know the value of the following sum: $$\\sum_{v,u,v\\neq u} d(i,v,u)$$.\n\nHelp Greg, print the value of the required sum before each s

In [21]:
import pandas as pd
import os

def merge_csv_columns(directory_path):
    all_data = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".csv"):
            filepath = os.path.join(directory_path, filename)
            try:
                df = pd.read_csv(filepath)
                # Select desired columns if they exist
                selected_columns = ['contest_id', 'index', 'statement']
                available_columns = [col for col in selected_columns if col in df.columns]

                if len(available_columns) > 0:
                    all_data.append(df[available_columns])
                else:
                    print(f"Warning: None of the required columns ('contest_id', 'index', 'statement') found in {filename}")

            except Exception as e:
                print(f"Error processing {filename}: {e}")

    if all_data:
        merged_df = pd.concat(all_data, ignore_index=True)
        return merged_df
    else:
        print("No valid CSV files found or no required columns found in any file.")
        return pd.DataFrame()

directory = '/content/drive/MyDrive/data_small_language_models/selected_problems'
merged_dataframe = merge_csv_columns(directory)
display(merged_dataframe.head())

,contest_id,index,statement
0,4,A,Description:\r\n\r\n\r\nOne hot summer day Pet...
1,71,A,Description:\r\n\r\n\r\nSometimes some words l...
2,231,A,Description:\r\n\r\n\r\nOne day three best fri...
3,282,A,Description:\r\n\r\n\r\nThe classic programmin...
4,158,A,"Description:\r\n\r\n\r\n""Contestant who earns ..."


In [24]:
merged_dataframe['statement'][1]

'Description:\r\n\r\n\r\nSometimes some words like " localization " or " internationalization " are so long that writing them many times in one text is quite tiresome.\r\n Let\'s consider a word too long , if its length is strictly more than 10 characters. All too long words should be replaced with a special abbreviation.\r\n This abbreviation is made like this: we write down the first and the last letter of a word and between them we write the number of letters between the first and the last letters. That number is in decimal system and doesn\'t contain any leading zeroes.\r\n Thus, " localization " will be spelt as " l10n ", and " internationalization » will be spelt as " i18n ".\r\n You are suggested to automatize the process of changing the words with abbreviations. At that all too long words should be replaced by the abbreviation and the words that are not too long should not undergo any changes.\r\n\r\nInput\r\n The first line contains an integer n ( 1\u2009≤ n ≤\u2009100 ). Each

In [26]:
import pandas as pd
import json
from google.colab import files

descriptions_df = merged_dataframe
print(f"\n✓ Loaded CSV with {len(descriptions_df)} rows")
print("\nColumns in CSV:")
print(descriptions_df.columns.tolist())
print("\nFirst few rows:")
print(descriptions_df.head())


print("\n" + "=" * 60)
print("Column Mapping")
print("=" * 60)

columns_lower = [c.lower() for c in descriptions_df.columns]

contest_col = None
for possible in ['contest_id', 'contestid', 'contest']:
    if possible in columns_lower:
        contest_col = descriptions_df.columns[columns_lower.index(possible)]
        break

index_col = None
for possible in ['index', 'problem', 'problem_letter', 'letter']:
    if possible in columns_lower:
        index_col = descriptions_df.columns[columns_lower.index(possible)]
        break

description_col = None
for possible in ['description', 'problem_statement', 'statement', 'text']:
    if possible in columns_lower:
        description_col = descriptions_df.columns[columns_lower.index(possible)]
        break

print("\nAuto-detected columns:")
print(f"  Contest ID: {contest_col}")
print(f"  Problem Index: {index_col}")
print(f"  Description: {description_col}")


if not contest_col or not index_col or not description_col:
    print("\n⚠ Could not auto-detect all columns. Please specify:")

    if not contest_col:
        print("\nEnter column name for contest_id:")
        contest_col = input().strip()

    if not index_col:
        print("\nEnter column name for problem index (e.g., A, B, C):")
        index_col = input().strip()

    if not description_col:
        print("\nEnter column name for description:")
        description_col = input().strip()

print(f"\n✓ Using columns:")
print(f"  Contest ID: {contest_col}")
print(f"  Problem Index: {index_col}")
print(f"  Description: {description_col}")


print("\n" + "=" * 60)
print("Creating Description Lookup")
print("=" * 60)

description_lookup = {}

for _, row in descriptions_df.iterrows():
    contest_id = str(row[contest_col]).strip()
    problem_index = str(row[index_col]).strip().upper()
    description = str(row[description_col]).strip()


    key1 = f"{contest_id}/{problem_index}"
    key2 = f"{contest_id}{problem_index}"

    description_lookup[key1] = description
    description_lookup[key2] = description

print(f"✓ Created lookup with {len(description_lookup)} entries")
print(f"\nSample keys: {list(description_lookup.keys())[:5]}")



print("\n" + "=" * 60)
print("Replacing Descriptions")
print("=" * 60)

# Assuming formatted_problems already exists from previous script
# If not, load it:
"""
with open('problems_with_testcases.json', 'r') as f:
    data = json.load(f)
    formatted_problems = data['problems']
"""

replaced_count = 0
not_found_count = 0

for problem in formatted_problems:
    problem_id = problem['problem_id']
    dataset_id = problem['dataset_id']

    new_description = None

    if dataset_id in description_lookup:
        new_description = description_lookup[dataset_id]
    elif problem_id in description_lookup:
        new_description = description_lookup[problem_id]
    else:
        normalized_dataset_id = dataset_id.replace('/', '')
        if normalized_dataset_id in description_lookup:
            new_description = description_lookup[normalized_dataset_id]

    if new_description:
        old_desc_preview = problem['problem_statement'][:50]
        problem['problem_statement'] = new_description
        replaced_count += 1
        print(f"✓ Replaced {problem_id}")

    else:
        not_found_count += 1
        print(f"✗ Not found in CSV: {problem_id}")

print(f"\n{'='*60}")
print("REPLACEMENT SUMMARY")
print('='*60)
print(f"✓ Replaced: {replaced_count}/{len(formatted_problems)}")
print(f"✗ Not found in CSV: {not_found_count}/{len(formatted_problems)}")


print("\n" + "=" * 60)
print("Saving Updated Data")
print("=" * 60)

# Save updated formatted_problems
output_data = {
    'metadata': {
        'total_problems': len(formatted_problems),
        'descriptions_replaced': replaced_count,
        'timestamp': pd.Timestamp.now().isoformat()
    },
    'problems': formatted_problems
}


with open('problems_with_custom_descriptions.json', 'w') as f:
    json.dump(output_data, f, indent=2)

print("✓ Saved to: problems_with_custom_descriptions.json")


if replaced_count > 0:
    print("\n" + "=" * 60)
    print("EXAMPLE: First Replaced Problem")
    print("=" * 60)

    for problem in formatted_problems:
        if problem['problem_id'] in description_lookup or problem['dataset_id'] in description_lookup:
            print(f"\nProblem ID: {problem['problem_id']}")
            print(f"New Description Preview:")
            print(problem['problem_statement'][:300] + "...")
            break

print("\n" + "=" * 60)
print("✓ DONE!")
print("=" * 60)
print(f"Variable 'formatted_problems' now has your custom descriptions")
print(f"Updated: {replaced_count} problems")


✓ Loaded CSV with 350 rows

Columns in CSV:
['contest_id', 'index', 'statement']

First few rows:
   contest_id index                                          statement
0           4     A  Description:\r\n\r\n\r\nOne hot summer day Pet...
1          71     A  Description:\r\n\r\n\r\nSometimes some words l...
2         231     A  Description:\r\n\r\n\r\nOne day three best fri...
3         282     A  Description:\r\n\r\n\r\nThe classic programmin...
4         158     A  Description:\r\n\r\n\r\n"Contestant who earns ...

Column Mapping

Auto-detected columns:
  Contest ID: contest_id
  Problem Index: index
  Description: statement

✓ Using columns:
  Contest ID: contest_id
  Problem Index: index
  Description: statement

Creating Description Lookup
✓ Created lookup with 700 entries

Sample keys: ['4/A', '4A', '71/A', '71A', '231/A']

Replacing Descriptions
✓ Replaced 295B
✓ Replaced 318A
✓ Replaced 617A
✓ Replaced 1971C
✓ Replaced 300C
✓ Replaced 519B
✓ Replaced 519E
✓ Replaced 110A
✓ R

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓ Downloaded!

✓ DONE!
Variable 'formatted_problems' now has your custom descriptions
Updated: 344 problems


In [27]:
formatted_problems[1]

{'problem_id': '318A',
 'dataset_id': '318/A',
 'problem_statement': "Description:\r\n\r\n\r\nBeing a nonconformist, Volodya is displeased with the current state of things, particularly with the order of natural numbers (natural number is positive integer number). He is determined to rearrange them. But there are too many natural numbers, so Volodya decided to start with the first n . He writes down the following sequence of numbers: firstly all odd integers from 1 to n (in ascending order), then all even integers from 1 to n (also in ascending order). Help our hero to find out which number will stand at the position number k .\r\n\r\nInput\r\n The only line of input contains integers n and k ( 1\u2009≤ k ≤ n ≤\u200910 12 ).\r\n Please, do not use the %lld specifier to read or write 64-bit integers in C++. It is preferred to use the cin , cout streams or the %I64d specifier.\r\n\r\nOutput\r\n Print the number that will stand at the position number k after Volodya's manipulations.\r\n\r

# Booting Ollama Server

In [29]:
import subprocess
import time
import requests


print("=" * 60)
print("Checking Ollama Installation")
print("=" * 60)

try:
    result = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
    print(f"✓ Ollama installed: {result.stdout.strip()}")
except FileNotFoundError:
    print("✗ Ollama not found. Installing...")
    !curl -fsSL https://ollama.com/install.sh | sh
    print("✓ Ollama installed!")


print("\n" + "=" * 60)
print("Starting Ollama Server")
print("=" * 60)


print("Cleaning up old processes...")
!pkill -9 ollama 2>/dev/null || true

time.sleep(2)


print("Starting Ollama server...")
ollama_process = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    start_new_session=True
)

print("✓ Server starting...")
print("  Waiting for server to be ready...")


max_retries = 30
for i in range(max_retries):
    try:
        response = requests.get('http://localhost:11434/api/tags', timeout=1)
        if response.status_code == 200:
            print(f"✓ Ollama server is ready! (took {i+1} seconds)")
            break
    except:
        pass

    time.sleep(1)

    if i % 5 == 0:
        print(f"  Still waiting... ({i+1}/{max_retries})")
else:
    print("✗ Server did not start in time. Check logs below:")
    print(ollama_process.stderr.read().decode())


print("\n" + "=" * 60)
print("Verifying Server Status")
print("=" * 60)

try:
    response = requests.get('http://localhost:11434/api/tags')
    print("✓ Server is responding!")
    print(f"  Status code: {response.status_code}")

    data = response.json()
    if 'models' in data and data['models']:
        print(f"\n✓ Available models:")
        for model in data['models']:
            print(f"  - {model['name']}")
    else:
        print("\n⚠ No models downloaded yet")

except Exception as e:
    print(f"✗ Server not responding: {e}")
    print("\nTroubleshooting:")
    print("  1. Run: !ollama serve")
    print("  2. Check logs: !cat ~/.ollama/logs/server.log")



print("\n" + "=" * 60)
print("Checking Model: phi4:14b")
print("=" * 60)

try:
    response = requests.get('http://localhost:11434/api/tags')
    models = response.json().get('models', [])
    model_names = [m['name'] for m in models]

    if 'phi4:14b' in model_names or 'phi4' in model_names:
        print("✓ phi4:14b is already downloaded")
    else:
        print("⚠ phi4:14b not found. Downloading...")
        print("  This will take 5-10 minutes (~8GB)...")
        !ollama pull phi4:14b
        print("✓ Model downloaded!")

except Exception as e:
    print(f"✗ Could not check models: {e}")



print("\n" + "=" * 60)
print("Testing LangChain Connection")
print("=" * 60)

try:
    from langchain_ollama import OllamaLLM

    model = OllamaLLM(model="phi4:14b")

    print("Testing model with simple prompt...")
    response = model.invoke("Print only the number 42")

    print(f"✓ Model responded!")
    print(f"  Response: {response[:100]}")

except Exception as e:
    print(f"✗ LangChain connection failed: {e}")
    print("\nTroubleshooting:")
    print("  1. Make sure Ollama server is running")
    print("  2. Try restarting: !pkill ollama && ollama serve &")
    print("  3. Check: requests.get('http://localhost:11434/api/tags')")



print("\n" + "=" * 60)
print("Server Management")
print("=" * 60)

print("""
✓ Ollama server is now running in background

To check server status:
  import requests
  requests.get('http://localhost:11434/api/tags').json()

If server stops, restart with:
  !pkill ollama
  !ollama serve &

Or use the cell above to restart automatically.

⚠ NOTE: In Colab, the server will stop if runtime disconnects.
   You'll need to re-run this cell after reconnecting.
""")

print("\n✓ Setup complete! You can now run your evaluation.")

Checking Ollama Installation
✓ Ollama installed: Warning: could not connect to a running Ollama instance

Starting Ollama Server
Cleaning up old processes...
Starting Ollama server...
✓ Server starting...
  Waiting for server to be ready...
  Still waiting... (1/30)
✓ Ollama server is ready! (took 2 seconds)

Verifying Server Status
✓ Server is responding!
  Status code: 200

✓ Available models:
  - phi4:14b

Checking Model: phi4:14b
✓ phi4:14b is already downloaded

Testing LangChain Connection
Testing model with simple prompt...
✓ Model responded!
  Response: ```
42
```

Server Management

✓ Ollama server is now running in background

To check server status:
  import requests
  requests.get('http://localhost:11434/api/tags').json()

If server stops, restart with:
  !pkill ollama
  !ollama serve &
  
Or use the cell above to restart automatically.

⚠ NOTE: In Colab, the server will stop if runtime disconnects.
   You'll need to re-run this cell after reconnecting.

QUICK COMMANDS

# C

# Generating and Evaluating Results on Test Cases

In [ ]:
import subprocess
import re
import json
from datetime import datetime

class CodeforcesSolver:
    def __init__(self, model_name="phi4:14b"):
        from langchain_ollama import OllamaLLM
        self.model = OllamaLLM(model=model_name)
        self.model_name = model_name

    def create_prompt(self, problem_statement, language="Python"):
        """Create prompt following the paper's methodology"""
        template = f"""You are a highly skilled competitive programmer with 15 years of experience in the field.

Your objective is to analyze the following problem statement and to produce a {language} code solution that adheres to the requirements.

Guidelines for the solution:
- Deliver only the {language} code
- Ensure the solution reads input via standard input and produces outputs via standard output
- If the solution requires defining a function, ensure it is executed within the code
- Avoid adding explanations, comments, or unnecessary text

The Problem Statement includes a detailed description, input and output format, and examples to clarify requirements.

Problem Statement:
{problem_statement}

Generate the {language} solution:"""
        return template

    def extract_code(self, model_output):
        """Extract code from model output"""
        pattern = r'```(?:python|cpp|c\+\+)?\n(.*?)```'
        matches = re.findall(pattern, model_output, re.DOTALL)
        if matches:
            return matches[0].strip()
        return model_output.strip()

    def generate_solutions(self, problem_statement, language="Python", num_attempts=3):
        """Generate multiple solutions (pass@k)"""
        prompt = self.create_prompt(problem_statement, language)
        solutions = []

        for i in range(num_attempts):
            print(f"    Generating solution {i+1}/{num_attempts}...", end=" ", flush=True)
            try:
                response = self.model.invoke(prompt)
                clean_code = self.extract_code(response)
                solutions.append(clean_code)
                print("✓")
            except Exception as e:
                print(f"✗ Error: {e}")
                solutions.append(None)

        return solutions

    def test_solution(self, code, test_cases, timeout=5):
        """Test solution with test cases"""
        if code is None:
            return [{'passed': False, 'error': 'No code generated'}]

        results = []

        for i, (test_input, expected_output) in enumerate(test_cases):
            with open('temp_solution.py', 'w') as f:
                f.write(code)

            try:
                process = subprocess.run(
                    ['python', 'temp_solution.py'],
                    input=test_input,
                    capture_output=True,
                    text=True,
                    timeout=timeout
                )

                actual_output = process.stdout.strip()
                expected = expected_output.strip()
                passed = actual_output == expected

                results.append({
                    'test_case': i+1,
                    'passed': passed,
                    'expected': expected,
                    'actual': actual_output,
                    'error': None
                })

            except subprocess.TimeoutExpired:
                results.append({
                    'test_case': i+1,
                    'passed': False,
                    'error': 'Time Limit Exceeded'
                })
            except Exception as e:
                results.append({
                    'test_case': i+1,
                    'passed': False,
                    'error': str(e)
                })

        return results


def evaluate_formatted_problems(formatted_problems, model_name="phi4:14b", num_attempts=3):
    """
    Evaluate problems from formatted_problems list

    Args:
        formatted_problems: List of problem dicts with 'problem_statement' and 'test_cases'
        model_name: Model to use (default: phi4:14b)
        num_attempts: Number of solutions per problem (default: 3 for pass@3)

    Returns:
        results_summary: List of results for each problem
    """

    # Initialize solver
    solver = CodeforcesSolver(model_name=model_name)

    results_summary = []

    print(f"{'='*60}")
    print(f"Starting Evaluation")
    print(f"Model: {model_name}")
    print(f"Problems: {len(formatted_problems)}")
    print(f"Attempts per problem: {num_attempts}")
    print(f"{'='*60}\n")

    for idx, problem in enumerate(formatted_problems, 1):
        print(f"\n{'='*60}")
        print(f"Problem {idx}/{len(formatted_problems)}: {problem['problem_id']}")
        print(f"Rating: {problem.get('rating', 'Unknown')}")
        print(f"Test cases: {problem['num_tests']}")
        print('='*60)

        # Generate solutions
        solutions = solver.generate_solutions(
            problem['problem_statement'],
            num_attempts=num_attempts
        )

        # Test all solutions (pass@k logic)
        all_test_results = []
        passed_any = False

        for sol_idx, solution in enumerate(solutions, 1):
            if solution is None:
                continue

            print(f"\n  Testing solution {sol_idx}/{num_attempts}...")

            # Test with all test cases
            test_results = solver.test_solution(solution, problem['test_cases'])
            all_test_results.append(test_results)

            # Check if all tests passed
            num_passed = sum(1 for r in test_results if r['passed'])
            all_passed = all(r['passed'] for r in test_results)

            print(f"    Tests passed: {num_passed}/{len(test_results)}", end="")

            if all_passed:
                print(" ✓ ACCEPTED")
                passed_any = True

            else:
                print(" ✗ FAILED")

        # Calculate pass@k for this problem
        pass_at_1 = len(all_test_results) > 0 and all(r['passed'] for r in all_test_results[0])
        pass_at_2 = len(all_test_results) > 1 and any(
            all(r['passed'] for r in results) for results in all_test_results[:2]
        )
        pass_at_3 = any(
            all(r['passed'] for r in results) for results in all_test_results
        )

        result = {
            'problem_id': problem['problem_id'],
            'rating': problem.get('rating', 'Unknown'),
            'num_tests': problem['num_tests'],
            'num_solutions_generated': len([s for s in solutions if s is not None]),
            'pass@1': pass_at_1,
            'pass@2': pass_at_2,
            'pass@3': pass_at_3,
            'timestamp': datetime.now().isoformat()
        }

        results_summary.append(result)

        print(f"\n  Result: {'✓ PASSED' if pass_at_3 else '✗ FAILED'} (pass@3)")

    # Overall Summary
    total = len(results_summary)
    passed_1 = sum(1 for r in results_summary if r['pass@1'])
    passed_2 = sum(1 for r in results_summary if r['pass@2'])
    passed_3 = sum(1 for r in results_summary if r['pass@3'])

    print(f"\n{'='*60}")
    print("EVALUATION SUMMARY")
    print('='*60)
    print(f"Model: {model_name}")
    print(f"Total problems: {total}")
    print(f"\nAccuracy:")
    print(f"  pass@1: {passed_1}/{total} ({passed_1/total*100:.1f}%)")
    print(f"  pass@2: {passed_2}/{total} ({passed_2/total*100:.1f}%)")
    print(f"  pass@3: {passed_3}/{total} ({passed_3/total*100:.1f}%)")
    print('='*60)


    output_data = {
        'model': model_name,
        'timestamp': datetime.now().isoformat(),
        'num_problems': total,
        'pass@1': f"{passed_1}/{total} ({passed_1/total*100:.1f}%)",
        'pass@2': f"{passed_2}/{total} ({passed_2/total*100:.1f}%)",
        'pass@3': f"{passed_3}/{total} ({passed_3/total*100:.1f}%)",
        'results': results_summary
    }

    with open('evaluation_results.json', 'w') as f:
        json.dump(output_data, f, indent=2)

    print("\n✓ Results saved to: evaluation_results.json")

    return results_summary

# load
"""
with open('problems_with_testcases.json', 'r') as f:
    data = json.load(f)
    formatted_problems = data['problems']
"""

# Run evaluation on all problems
results = evaluate_formatted_problems(
    formatted_problems,
    model_name="phi4:14b",
    num_attempts=3  # pass@3
)


from google.colab import files
files.download('evaluation_results.json')

print("\n✓ Evaluation complete!")

Starting Evaluation
Model: phi4:14b
Problems: 344
Attempts per problem: 3


Problem 1/344: 295B
Rating: 1700
Test cases: 6
    Generating solution 1/3... ✓
    Generating solution 2/3... ✓
    Generating solution 3/3... ✓

  Testing solution 1/3...
    Tests passed: 1/6 ✗ FAILED

  Testing solution 2/3...
    Tests passed: 1/6 ✗ FAILED

  Testing solution 3/3...
    Tests passed: 1/6 ✗ FAILED

  Result: ✗ FAILED (pass@3)

Problem 2/344: 318A
Rating: 900
Test cases: 25
    Generating solution 1/3... ✓
    Generating solution 2/3... ✓
    Generating solution 3/3... ✓

  Testing solution 1/3...
    Tests passed: 25/25 ✓ ACCEPTED

  Testing solution 2/3...
    Tests passed: 25/25 ✓ ACCEPTED

  Testing solution 3/3...
    Tests passed: 25/25 ✓ ACCEPTED

  Result: ✓ PASSED (pass@3)

Problem 3/344: 617A
Rating: 800
Test cases: 34
    Generating solution 1/3... ✓
    Generating solution 2/3... ✓
    Generating solution 3/3... ✓

  Testing solution 1/3...
    Tests passed: 34/34 ✓ ACCEPTED

  T

# Result Analysis

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

with open("evaluation_results.json", "r") as f:
    data = json.load(f)

results = pd.DataFrame(data["results"])

for col in ["pass@1", "pass@2", "pass@3"]:
    results[col] = results[col].astype(int)


print("Model:", data["model"])
print("Evaluated problems:", len(results))
print("\nAverage Accuracy:")
print(results[["pass@1", "pass@2", "pass@3"]].mean() * 100)


acc = results[["pass@1", "pass@2", "pass@3"]].mean() * 100
plt.figure(figsize=(6,4))
sns.barplot(x=acc.index, y=acc.values, palette="coolwarm")
plt.title("Overall Pass@k Accuracy (%)")
plt.ylabel("Accuracy (%)")
plt.xlabel("Metric")
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


results["rating"] = pd.to_numeric(results["rating"], errors="coerce")
rating_stats = results.groupby("rating")[["pass@1", "pass@2", "pass@3"]].mean() * 100

plt.figure(figsize=(8,5))
sns.lineplot(data=rating_stats, markers=True, dashes=False)
plt.title("Pass@k Accuracy by Problem Rating")
plt.xlabel("Problem Rating")
plt.ylabel("Accuracy (%)")
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
